# Car Price Prediction Notebook

This notebook builds a regression model to predict car selling prices using the `car data.csv` dataset. It includes data loading, preprocessing, feature engineering, model training, evaluation, and real-world insights.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)

## Load and Inspect the Dataset

In [2]:
# Load the car dataset
base_path = Path(r'c:\Users\Mudassar\Desktop\CodeAlpha-Data Science\Car_Price_Prediction')
file_path = base_path / 'car data.csv'

df = pd.read_csv(file_path)
print('Dataset shape:', df.shape)
print('\nColumns:', df.columns.tolist())

df.head()

Dataset shape: (301, 9)

Columns: ['Car_Name', 'Year', 'Selling_Price', 'Present_Price', 'Driven_kms', 'Fuel_Type', 'Selling_type', 'Transmission', 'Owner']


,Car_Name,Year,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


In [3]:
# Inspect data types and missing values
print(df.info())
print('\nMissing values per column:\n', df.isna().sum())

print('\nUnique values for categorical columns:')
for col in ['Fuel_Type', 'Selling_type', 'Transmission', 'Owner']:
    print(f"\n{col}: {df[col].unique()}")

<class 'pandas.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Car_Name       301 non-null    str    
 1   Year           301 non-null    int64  
 2   Selling_Price  301 non-null    float64
 3   Present_Price  301 non-null    float64
 4   Driven_kms     301 non-null    int64  
 5   Fuel_Type      301 non-null    str    
 6   Selling_type   301 non-null    str    
 7   Transmission   301 non-null    str    
 8   Owner          301 non-null    int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 21.3 KB
None

Missing values per column:
 Car_Name         0
Year             0
Selling_Price    0
Present_Price    0
Driven_kms       0
Fuel_Type        0
Selling_type     0
Transmission     0
Owner            0
dtype: int64

Unique values for categorical columns:

Fuel_Type: <StringArray>
['Petrol', 'Diesel', 'CNG']
Length: 3, dtype: str

Selling_type: <StringArray>
[

## Feature Engineering and Preprocessing

In [7]:
# Feature engineering: car age and brand goodwill
current_year = 2020

df = df.copy()
df['Car_Age'] = current_year - df['Year']

def extract_brand(name):
    return name.split()[0].lower()

df['Brand'] = df['Car_Name'].apply(extract_brand)

# Keep key features and the target
features = ['Present_Price', 'Driven_kms', 'Fuel_Type', 'Selling_type', 'Transmission', 'Owner', 'Car_Age', 'Brand']
target = 'Selling_Price'

print('Engineered features sample:')
print(df[['Car_Name', 'Brand', 'Year', 'Car_Age', 'Present_Price', 'Driven_kms']].head())

Engineered features sample:
  Car_Name  Brand  Year  Car_Age  Present_Price  Driven_kms
0     ritz   ritz  2014        6           5.59       27000
1      sx4    sx4  2013        7           9.54       43000
2     ciaz   ciaz  2017        3           9.85        6900
3  wagon r  wagon  2011        9           4.15        5200
4    swift  swift  2014        6           6.87       42450


### Preprocessing Strategy

- Numeric features: `Present_Price`, `Driven_kms`, `Car_Age`
- Categorical features: `Fuel_Type`, `Selling_type`, `Transmission`, `Owner`, `Brand`
- Use one-hot encoding for categorical variables and scaling for numeric columns.

In [ ]:
# Split features and target
X = df[features]
y = df[target]

numeric_features = ['Present_Price', 'Driven_kms', 'Car_Age']
cat_features = ['Fuel_Type', 'Selling_type', 'Transmission', 'Owner', 'Brand']

numeric_transformer = Pipeline([
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', cat_transformer, cat_features)
    ]
)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

TypeError: OneHotEncoder.__init__() got an unexpected keyword argument 'sparse'

## Train Regression Models

In [6]:
# Create pipelines for Linear Regression and Random Forest
models = {
    'Linear Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', LinearRegression())
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
    ])
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    print(f"{name} results:")
    print(f"  MAE: {mae:.3f}")
    print(f"  RMSE: {rmse:.3f}")
    print(f"  R2: {r2:.3f}\n")

NameError: name 'preprocessor' is not defined

## Model Evaluation and Visualization

In [ ]:
# Compare actual vs predicted values for the best model
best_model_name = max(results, key=lambda name: results[name]['R2'])
best_model = models[best_model_name]

y_pred_best = best_model.predict(X_test)

plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_best, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--')
plt.title(f'Actual vs Predicted Selling Price ({best_model_name})')
plt.xlabel('Actual Selling Price (Lakhs)')
plt.ylabel('Predicted Selling Price (Lakhs)')
plt.tight_layout()
plt.show()

residuals = y_test - y_pred_best
plt.figure(figsize=(10, 5))
plt.hist(residuals, bins=20, edgecolor='k')
plt.title('Residual Distribution')
plt.xlabel('Residual (Actual - Predicted)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

## Feature Importance and Business Insights

In [ ]:
# Compute feature importance for Random Forest model if used
if best_model_name == 'Random Forest':
    rf = best_model.named_steps['regressor']
    feature_names = numeric_features + list(models['Random Forest'].named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_features))
    importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
    print('Top 10 feature importances:')
    print(importances.head(10))
    
    plt.figure(figsize=(10, 6))
    importances.head(10).plot(kind='bar')
    plt.title('Top 10 Feature Importances')
    plt.ylabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    print('Feature importance visualization is available for tree-based models.')

## Real-World Applications of Car Price Prediction

This workflow demonstrates how machine learning can support:

- Used car valuation platforms that estimate fair resale prices.
- Dealership pricing strategies to balance inventory and revenue.
- Customer decision-making through transparent price estimates.
- Financial lending and insurance risk assessment using vehicle value prediction.

## Conclusion

The notebook shows how to load auto sales data, engineer features like brand goodwill and car age, preprocess categorical and numeric fields, train regression models, and evaluate price prediction performance with real-world business relevance.